In [ ]:
import os
import numpy as np
from ase import units
from ase.db import connect
from ase.calculators.orca import ORCA, OrcaProfile
from pyscf import gto, dft, hessian, lib
from pyscf.geomopt.geometric_solver import optimize


In [ ]:
#dataset_path = "/Users/joaopanicali/SLAC_v2/SLAC-atoms/train_4M" # using 4m dataset for exploration. Much smaller than the whole dataset
#dataset = AseDBDataset({"src": dataset_path})
#print(f"Total molecules in this split: {len(dataset)}")
data_dir = os.path.expanduser("./train_4M")
shards = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith(".aselmdb")]
print(f"Found {len(shards)} shards")
db = connect(shards[0])

In [ ]:
#query molecule with specific number of atoms
for row in db.select('natoms=3,H=2,O=1', limit=1):
    dataset_index = row.id - 1    
    print(f"Found {row.formula} natively at database ID {row.id} (Dataset index: {dataset_index})")
    atoms = row.toatoms()
    atoms.info["charge"] = row.data.get("charge", 0)
    atoms.info["spin"] = row.data.get("spin", 1)

In [ ]:
def compute_and_save_modes_pyscf(
    atoms, 
    output_npz_path: str, 
    opt: bool = False, 
    functional: str = 'b3lyp', # 'wb97m'
    dispersion: str = "d4", #None
    basis: str = 'def2-tzvpd',
    freq_lb: float = 5.0, 
    freq_ub: float = 4000.0
):
    """
    Runs an in-memory DFT calculation (with optional geometry optimization) 
    and analytical Hessian calculation using PySCF.
    """
    # 1. Molecule Setup
    atom_str = "; ".join([f"{atom.symbol} {atom.x} {atom.y} {atom.z}" for atom in atoms])
    lib.num_threads(16) # set number of cores for each molecule. 
    # Assuming your dataset stores Multiplicity under the "spin" key (defaulting to singlet = 1)
    spin_multiplicity = atoms.info.get("spin", 1)
    pyscf_spin = spin_multiplicity - 1 # PySCF needs the number of unpaired electrons
    charge = atoms.info.get("charge", 0)

    mol = gto.M(
        atom=atom_str,
        basis=basis,
        charge=charge,
        spin=pyscf_spin,
        verbose=0
    )

    # 2. Dynamic Calculator Setup (Robustness for diverse molecules)
    # Open-shell molecules (radicals/triplets) require Unrestricted Kohn-Sham (UKS)
    # Closed-shell molecules (singlets) use Restricted Kohn-Sham (RKS) for speed
    def build_calculator(mol_obj):
        mf = dft.UKS(mol_obj) if pyscf_spin > 0 else dft.RKS(mol_obj)
        mf.xc = functional
        if dispersion is not None:
            mf.disp = dispersion
        
        # Stability parameters for large/complex molecules
        mf.max_cycle = 300 
        mf.conv_tol = 1e-8 
        #if GPU_AVAILABLE:
        #    print("GPU detected: Offloading math to VRAM.") # Optional logging
        #    return mf.to_gpu()
        #else:
        #    print("No GPU detected: Running entirely on CPU RAM.") # Optional logging
        return mf

    # 3. Optional Geometry Optimization
    if opt:
        print(f"Running geometry optimization with {functional} and {dispersion}...")
        mf_opt = build_calculator(mol)
        mol_final = optimize(mf_opt)
    else:
        print("Skipping geometry optimization. Using initial coordinates...")
        mol_final = mol

    # 4. Analytical Hessian Calculation
    print(f"Computing analytical Hessian with {functional} and {dispersion}...")
    mf_final = build_calculator(mol_final)
    mf_final.kernel() # Solve SCF
    
    # Calculate Hessian: 4D array (N_atoms, N_atoms, 3, 3)
    raw_hessian_4d = mf_final.Hessian().kernel()
    
    # Reshape the 4D array into the standard 2D 3N x 3N symmetric matrix
    N = mol_final.natm
    raw_hessian_2d = raw_hessian_4d.transpose(0, 2, 1, 3).reshape(3*N, 3*N)

    # 5. Process Physics (ASE Standard Units)
    conversion_factor = units.Hartree / (units.Bohr ** 2)
    hessian_ASE = raw_hessian_2d * conversion_factor
    
    # Extract coordinates in Angstroms (PySCF internal coords are in Bohr, need conversion)
    R_0 = mol_final.atom_coords() * units.Bohr 
    masses_amu = np.repeat([mol_final.atom_mass_list()], 3)
    
    M_inv_sqrt = np.diag(1.0 / np.sqrt(masses_amu))
    tilde_H_ASE = M_inv_sqrt @ hessian_ASE @ M_inv_sqrt
    
    eigenvalues, V = np.linalg.eigh(tilde_H_ASE)

    n_atoms = len(atoms)
    if n_atoms == 1:
        n_rigid = 3
    elif n_atoms == 2:
        n_rigid = 5
    else:
        n_rigid = 6

    eigenvalues_trunc = eigenvalues[n_rigid:]
    V = V[:, n_rigid:]

    # Prevent NaN crashes from numerical noise (-1e-8)
    omegas_ASE = np.sqrt(np.abs(eigenvalues_trunc)) 
    raw_freqs_cm1 = omegas_ASE * 521.47

    # Filter frequencies
    valid_indices = np.where(
        (raw_freqs_cm1 > freq_lb) & 
        (raw_freqs_cm1 < freq_ub)
    )[0]
    
    omegas_ASE_filtered = omegas_ASE[valid_indices]
    V_vib_filtered = V[:, valid_indices]
    # 6. Save output
    os.makedirs(os.path.dirname(output_npz_path) or '.', exist_ok=True)
    np.savez_compressed(
        output_npz_path,
        R_0=R_0.flatten(),
        masses_amu=masses_amu,
        omegas_ASE=omegas_ASE_filtered,
        V_vib=V_vib_filtered,
        eigen_val = eigenvalues
    )
    print(f"Physical mode data successfully saved to {output_npz_path}")



In [ ]:
def generate_dynamic_trajectory(npz_path: str, T_kelvin: float = 300.0, points_per_fast_cycle: int = 10) -> tuple[np.ndarray, np.ndarray]:
    """
    This function generates trajectories. It determines the time delta to ensure that at least
    points_per_fast_cycle is sampled from highest frequency normal mode and the duration of the trajectory
    guarantees that we sample a full motion of the slowest mode.

    Outputs a time vector and coordinate matrix
    """
    
    data = np.load(npz_path)
    R_0 = data['R_0']
    masses_amu = data['masses_amu']
    omegas_ASE = data['omegas_ASE']
    V_vib = data['V_vib']
    
    num_modes = len(omegas_ASE)
    N_atoms = len(R_0) // 3
    
    # get periods in femptoseconds
    periods_fs = (2 * np.pi / omegas_ASE) / units.fs
    
    # get period of slowest vibration
    t_total_fs = np.max(periods_fs)  
    
    # define time step such that at least the defined number of points is sampled from the fastest normal mode's full motion
    dt_fs = np.min(periods_fs) / points_per_fast_cycle 
    
    time_grid_fs = np.arange(0, t_total_fs + dt_fs, dt_fs)
    num_frames = len(time_grid_fs)
    
    # Convert back to ASE units for the math engine
    time_steps_ASE = (time_grid_fs * units.fs).reshape(-1, 1) 
    
    print(f"--- Sampling Dynamics ---")
    print(f"Slowest Mode: {np.max(periods_fs):.2f} fs")
    print(f"Fastest Mode: {np.min(periods_fs):.2f} fs")
    print(f"Generated {num_frames} frames with dt = {dt_fs:.2f} fs")
    
    # get amplitude
    A_q = np.sqrt(2 * units.kB * T_kelvin) / omegas_ASE 
    
    # Random initial phases
    phi = np.random.uniform(0, 2 * np.pi, num_modes) 
    
    #calculating oscillaiton of mode
    c_t = A_q * np.cos(omegas_ASE * time_steps_ASE + phi) 
    
     
    # calculate mass-weighted displacements
    q_t = V_vib @ c_t.T 
    
    # convert back to cartesian coordinates
    M_inv_sqrt = np.diag(1.0 / np.sqrt(masses_amu))
    delta_R_Angstroms = M_inv_sqrt @ q_t 
    
    # extract final positions
    R_t = R_0[:, None] + delta_R_Angstroms 
    trajectory_tensor = R_t.T.reshape(num_frames, N_atoms, 3)
    
    return time_grid_fs, trajectory_tensor

In [ ]:
%%time
output_physics_file = "./mol_data/mol_001_physics.npz"
compute_and_save_modes_pyscf(
    atoms=atoms,
    opt = True,
    functional = 'b3lyp', # 'wb97m'
    dispersion = "d4",
    output_npz_path=output_physics_file
)

In [ ]:
generate_dynamic_trajectory(output_physics_file)

Code below is for ORCA

In [ ]:
def parse_orca_hessian(filepath: str) -> np.ndarray:
    """Parses an ORCA .hess file and extracts the symmetric Hessian matrix."""
    with open(filepath, 'r') as f:
        lines = f.readlines()

    start_line = next((i + 2 for i, line in enumerate(lines) if "$hessian" in line), None)
    if start_line is None:
        raise ValueError("Hessian section not found in the .hess file.")

    dim = int(lines[start_line - 1].strip())
    hessian = np.zeros((dim, dim))
    i = start_line
    
    while i < len(lines):
        parts = lines[i].split()
        if not parts or "$" in lines[i]: 
            break
        if not any("." in p for p in parts):
            col_start = int(parts[0])
        else:
            row_idx = int(parts[0])
            for j, val in enumerate(map(float, parts[1:])):
                hessian[row_idx, col_start + j] = val
        i += 1

    return (hessian + hessian.T) / 2

In [ ]:
def compute_and_save_modes(atoms, orca_path: str, output_npz_path: str, scratch_dir: str = "./orca_scratch",
 freq_lb: float = 5.0, freq_ub: float = 5000.0):
    """
    Runs ORCA, extracts the Hessian, calculates normal modes using ASE Standard Units, 
    and saves the physics data to disk for rapid trajectory generation.
    """
    os.makedirs(scratch_dir, exist_ok=True)
    base_name = os.path.join(scratch_dir, "mol_opt_freq")
    nprocs = int(os.environ.get('SLURM_NODES', os.cpu_count() or 4)) ##set number of corse used for each molecule
    atoms.calc = ORCA(
        profile=OrcaProfile(command=orca_path),
        directory="./orca_scratch",  
        label="mol_opt_freq",        
        orcasimpleinput= "B3LYP D4 def2-TZVP TightSCF TightOpt FREQ", #r2SCAN-3c Opt Freq'B3LYP D4 def2-TZVP TightSCF Opt FREQ', #include OPT?
        
        # change orca blocks for parallel processing
        orcablocks= f'%pal nprocs {nprocs} end\n %maxcore 4000\n%scf MaxIter 300 end', #f'%pal nprocs {nprocs} end\n%maxcore 4000\n%scf MaxIter 300 end',
        
        charge=atoms.info["charge"],
        mult=atoms.info["spin"], )
    
    atoms.get_potential_energy() 
    
    R_0 = atoms.get_positions().flatten() 
    masses_amu = np.repeat(atoms.get_masses(), 3) 
    raw_hessian = parse_orca_hessian(f"{scratch_dir}/orca.hess")
    
    # ORCA outputs Hartree/Bohr^2. ASE converts this to eV/Angstrom^2
    conversion_factor = units.Hartree / (units.Bohr ** 2)
    hessian_ASE = raw_hessian * conversion_factor
    
    # 4. Mass-Weighting and Eigendecomposition
    M_inv_sqrt = np.diag(1.0 / np.sqrt(masses_amu))
    tilde_H_ASE = M_inv_sqrt @ hessian_ASE @ M_inv_sqrt
    
    eigenvalues, V = np.linalg.eigh(tilde_H_ASE)

    n_atoms = len(atoms)
    if n_atoms == 1:
        n_rigid = 3
    elif n_atoms == 2:
        n_rigid = 5
    else:
        n_rigid = 6

    eigenvalues_trunc = eigenvalues[n_rigid:]
    V = V[:, n_rigid:]

    # Prevent NaN crashes from numerical noise (-1e-8)
    omegas_ASE = np.sqrt(np.abs(eigenvalues_trunc)) 
    raw_freqs_cm1 = omegas_ASE * 521.47

    # Filter frequencies
    valid_indices = np.where(
        (raw_freqs_cm1 > freq_lb) & 
        (raw_freqs_cm1 < freq_ub)
    )[0]
    
    omegas_ASE_filtered = omegas_ASE[valid_indices]
    V_vib_filtered = V[:, valid_indices]
    # 6. Save output
    os.makedirs(os.path.dirname(output_npz_path) or '.', exist_ok=True)
    np.savez_compressed(
        output_npz_path,
        R_0=R_0, #initial positions
        masses_amu=masses_amu, #molecule masses
        omegas_ASE=omegas_ASE_filtered, #mode frequencies (eigenvalues)
        V_vib=V_vib_filtered, # normal modes (eigen vectors)
        eigen = eigenvalues
    )
    print(f"Physical mode data successfully saved to {output_npz_path}")

In [ ]:
%%time
orca_command_path = "/home/cme290/joaop/SLAC-atoms/orca_6_1_1_linux_x86-64_shared_openmpi418_nodmrg/orca"
output_physics_file = "./mol_data/mol_002_physics.npz"
compute_and_save_modes(
    atoms=atoms, 
    orca_path=orca_command_path, 
    output_npz_path=output_physics_file,
)

In [ ]:
generate_dynamic_trajectory(output_physics_file)